In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

In [11]:
df = pd.read_csv("../Data/values.csv")
print ("Dataset Shape:", df.shape)
print ("\nColums:", df.columns.tolist())
df.head()


Dataset Shape: (180, 14)

Colums: ['patient_id', 'slope_of_peak_exercise_st_segment', 'thal', 'resting_blood_pressure', 'chest_pain_type', 'num_major_vessels', 'fasting_blood_sugar_gt_120_mg_per_dl', 'resting_ekg_results', 'serum_cholesterol_mg_per_dl', 'oldpeak_eq_st_depression', 'sex', 'age', 'max_heart_rate_achieved', 'exercise_induced_angina']


,patient_id,slope_of_peak_exercise_st_segment,thal,resting_blood_pressure,chest_pain_type,num_major_vessels,fasting_blood_sugar_gt_120_mg_per_dl,resting_ekg_results,serum_cholesterol_mg_per_dl,oldpeak_eq_st_depression,sex,age,max_heart_rate_achieved,exercise_induced_angina
0,0z64un,1,normal,128,2,0,0,2,308,0.0,1,45,170,0
1,ryoo3j,2,normal,110,3,0,0,0,214,1.6,0,54,158,0
2,yt1s1x,1,normal,125,4,3,0,2,304,0.0,1,77,162,1
3,l2xjde,1,reversible_defect,152,4,0,0,0,223,0.0,1,40,181,0
4,oyt4ek,3,reversible_defect,178,1,0,0,2,270,4.2,1,59,145,0


In [12]:
# PART 1: CATEGORICAL FEATURE ANALYSIS
categorical_cols = df.select_dtypes(include=['object','category']).columns
print("Categroical Colums:",categorical_cols)


Categroical Colums: Index(['patient_id', 'thal'], dtype='object')


In [ ]:
# PART 2: 'THAL' FEATURE ANALYSIS & ENCODING
print("\n" + "="*60)
print("ANALYSIS FOR 'THAL' FEATURE")
print("="*60)

print("\n--- Distribution Analysis ---")
thal_counts = df['thal'].value_counts()
thal_percent = df['thal'].value_counts(normalize=True) * 100
thal_stats = pd.DataFrame({
    'Count': thal_counts,
    'Percentage': thal_percent
})
print(thal_stats)

#Analyse: One-hot vs Ordinal Encodinf for thal
print("\n--- Encoding Strategy Analysis ---")
print("""
THAL CATEGORIES ANALYSIS:
- normal: Normal blood flow
- fixed_defect: Fixed perfusion defect
- reversible_defect: Reversible perfusion defect

ENCODING RECOMMENDATION:
1. ONE-HOT ENCODING (Preferred):
   - No inherent ordinal relationship between defect types
   - Each category is qualitatively different
   - Prevents model from assuming fixed_defect > reversible_defect > normal
   - Better preserves medical meaning

2. ORDINAL ENCODING (Alternative):
   - Could impose artificial order that doesn't exist medically
   - Might mislead models to think some categories are "worse" than others
   - Only use if there's a clear severity progression (which doesn't exist here)

CONCLUSION: One-hot encoding is recommended for 'thal'.
""")



ANALYSIS FOR 'THAL' FEATURE

--- Distribution Analysis ---
                   Count  Percentage
thal                                
normal                98   54.444444
reversible_defect     74   41.111111
fixed_defect           8    4.444444

--- Encoding Strategy Analysis ---

THAL CATEGORIES ANALYSIS:
- normal: Normal blood flow
- fixed_defect: Fixed perfusion defect
- reversible_defect: Reversible perfusion defect

ENCODING RECOMMENDATION:
1. ONE-HOT ENCODING (Preferred):
   - No inherent ordinal relationship between defect types
   - Each category is qualitatively different
   - Prevents model from assuming fixed_defect > reversible_defect > normal
   - Better preserves medical meaning

2. ORDINAL ENCODING (Alternative):
   - Could impose artificial order that doesn't exist medically
   - Might mislead models to think some categories are "worse" than others
   - Only use if there's a clear severity progression (which doesn't exist here)

CONCLUSION: One-hot encoding is recommend

In [ ]:
print("\n" + "="*40)
print("Method 1: One-Hot Encoding")
print("="*40)

df_onehot = df.copy()
thal_dummies = pd.get_dummies(df['thal'],prefix='thal', drop_first= False)

#adding the dummies to the dataframe (thal colum remains)
df_onehot = pd.concat([df_onehot,thal_dummies], axis= 1)

print("\n---- one-hot encoding results ---")
thal_onehot_cols = [col for col in df_onehot.columns if col.startswith('thal_')]
print("One-hot encoded columns:", thal_onehot_cols)

print("\nFirst 5 rows:")
print(df_onehot[['thal'] + thal_onehot_cols].head())



Method 1: One-Hot Encoding

---- one-hot encoding results ---
One-hot encoded columns: ['thal_fixed_defect', 'thal_normal', 'thal_reversible_defect']

First 5 rows:
                thal  thal_fixed_defect  thal_normal  thal_reversible_defect
0             normal              False         True                   False
1             normal              False         True                   False
2             normal              False         True                   False
3  reversible_defect              False        False                    True
4  reversible_defect              False        False                    True


In [16]:
print("\n" + "="*40)
print("Method 2: Ordinal Encoding")
print("="*40)

df_ordinal = df.copy()
encoder = OrdinalEncoder(categories= [['normal', 'fixed_defect', 'reversible_defect']])
df_ordinal['thal_ordinal'] = encoder.fit_transform(df[['thal']])

print("\n--- Oridnal Encoding results ---")
print("Encodinf Mapping:")
for i, category in enumerate(encoder.categories_[0]):
    print(f"{category} -> {i}")

print("\nFirst 5 rows:")
print(df_ordinal[['thal', 'thal_ordinal']].head())



Method 2: Ordinal Encoding

--- Oridnal Encoding results ---
Encodinf Mapping:
normal -> 0
fixed_defect -> 1
reversible_defect -> 2

First 5 rows:
                thal  thal_ordinal
0             normal           0.0
1             normal           0.0
2             normal           0.0
3  reversible_defect           2.0
4  reversible_defect           2.0


In [18]:
# Handle rare categories
print("\n---Rare Category Analysis ---")
threshold = 0.05 * len(df)
rare_categories = thal_counts[thal_counts < threshold].index.tolist()

thal_counts = df['thal'].value_counts()
rare_thal = thal_counts[thal_counts < threshold].index

if rare_categories:
    print(f"Rare Categories (<{threshold} occurrences): {rare_categories}")
    df['thal_cleaned'] = df['thal'].replace(rare_thal, 'Other')
    print("After cleaning rare categories:")
    print(df['thal_cleaned'].value_counts())
else:
    print("No Rare categories found (all categories have sufficient samples)")
    df['thal_cleaned'] = df['thal']



---Rare Category Analysis ---
Rare Categories (<9.0 occurrences): ['fixed_defect']
After cleaning rare categories:
thal_cleaned
normal               98
reversible_defect    74
Other                 8
Name: count, dtype: int64


In [19]:
# PART 3: 'CHEST_PAIN_TYPE' ANALYSIS & ENCODING
print("\n" + "="*60)
print("ANALYSIS FOR 'CHEST_PAIN_TYPE' FEATURE")
print("="*60)

# Analyse Distribution
print("\n--- Distribution Analysis ---")
chest_pain_counts = df['chest_pain_type'].value_counts()
print(chest_pain_counts)

# Determine if ordinal Rrelation exists
print("\n--- Ordinal Relationship Analysis ---")
print("""
CHEST PAIN TYPE MEDICAL ANALYSIS:
1. typical angina (value 1): Predictable chest pain, least severe
2. atypical angina (value 2): Less predictable, moderate severity  
3. non-anginal pain (value 3): Non-cardiac origin, moderate severity
4. asymptomatic (value 4): No symptoms but disease present - MOST SEVERE

MEDICAL SEVERITY PROGRESSION:
asymptomatic (4) > non-anginal (3) > atypical angina (2) > typical angina (1)

ORDINAL RELATIONSHIP EXISTS: Yes
- Clear progression from least severe (typical angina) to most severe (asymptomatic)
- Higher values indicate worse clinical presentation
- Appropriate for ordinal encoding based on medical severity
""")


ANALYSIS FOR 'CHEST_PAIN_TYPE' FEATURE

--- Distribution Analysis ---
chest_pain_type
4    82
3    57
2    28
1    13
Name: count, dtype: int64

--- Ordinal Relationship Analysis ---

CHEST PAIN TYPE MEDICAL ANALYSIS:
1. typical angina (value 1): Predictable chest pain, least severe
2. atypical angina (value 2): Less predictable, moderate severity  
3. non-anginal pain (value 3): Non-cardiac origin, moderate severity
4. asymptomatic (value 4): No symptoms but disease present - MOST SEVERE

MEDICAL SEVERITY PROGRESSION:
asymptomatic (4) > non-anginal (3) > atypical angina (2) > typical angina (1)

ORDINAL RELATIONSHIP EXISTS: Yes
- Clear progression from least severe (typical angina) to most severe (asymptomatic)
- Higher values indicate worse clinical presentation
- Appropriate for ordinal encoding based on medical severity



In [20]:
# Implement appropriate ordinal encoding with medical justification

# Medical sevrity-based mapping (most sever = highest number)
chest_pain_mapping = {
    1 : 0, # Typical anagina - LEAST servere
    2 : 1, # Atypical anagina
    3 : 2, # Non-anginal pain 
    4 : 3  # Asymptomatic - MOST servere
}

df['chest_pain_encoded'] = df['chest_pain_type'].map(chest_pain_mapping)

# Verify Encoding
print("Encoding Mapping:")
for orig, encoded in chest_pain_mapping.items():
    pain_type = {
        1: 'typical angina',
        2: 'atypical angina', 
        3: 'non-anginal pain',
        4: 'asymptomatic'
    }[orig]
    severity = ['Least Severe', 'Moderate', 'Moderate', 'Most Severe'][encoded]
    print(f"  {orig} ({pain_type}) -> {encoded} ({severity})")

print("\nVerification (first 10 rows):")
print(df[['chest_pain_type', 'chest_pain_encoded']].head(10))

Encoding Mapping:
  1 (typical angina) -> 0 (Least Severe)
  2 (atypical angina) -> 1 (Moderate)
  3 (non-anginal pain) -> 2 (Moderate)
  4 (asymptomatic) -> 3 (Most Severe)

Verification (first 10 rows):
   chest_pain_type  chest_pain_encoded
0                2                   1
1                3                   2
2                4                   3
3                4                   3
4                1                   0
5                3                   2
6                4                   3
7                4                   3
8                4                   3
9                3                   2


In [21]:
# PART 4: BINARY FEATURES ENCODING
print("\n" + "="*60)
print("BINARY FEATURES ANALYSIS")
print("="*60)

# Identify binary features from the dataset
print("\n--- Binary Feature Identification ---")
print("Checking candidate binary columns:")

# Check all potential binary columns
potential_binary = ['sex', 'fasting_blood_sugar_gt_120_mg_per_dl', 'exercise_induced_angina']
binary_cols = []

for col in potential_binary:
    if col in df.columns:
        unique_vals = sorted(df[col].dropna().unique())
        if len(unique_vals) <= 2:
            binary_cols.append(col)
            print(f' {col}: {unique_vals} (Binary - {len(unique_vals)} unique values)')
        else: 
            print(f" {col}: {unique_vals} (not binary - {len(unique_vals)} unique values)")
    else:
        print(f"{col}: Column not found in dataset")

print(f"\nConfirmed binary columns: {binary_cols}")


BINARY FEATURES ANALYSIS

--- Binary Feature Identification ---
Checking candidate binary columns:
 sex: [0, 1] (Binary - 2 unique values)
 fasting_blood_sugar_gt_120_mg_per_dl: [0, 1] (Binary - 2 unique values)
 exercise_induced_angina: [0, 1] (Binary - 2 unique values)

Confirmed binary columns: ['sex', 'fasting_blood_sugar_gt_120_mg_per_dl', 'exercise_induced_angina']


In [22]:
# Ensure 0/1 encoding consistency
print("\n--- Ensuring 0/1 Encoding Consistency ---")

for col in binary_cols:
    original_values = sorted(df[col].unique())
    
    if set(original_values) == {0,1}:
        print(f"{col}: Already encoded as 0/1")
    else:
        # convert to 0/1
        print(f"{col}: Converting {original_values} to 0/1")
        
        if col == 'sex':
            # Typically: 1 = male, 0 = female (or vice versa)
            # Keep exisiting mapping if it's already binary
            if set(original_values) <= {0,1}:
                df[col] = (df[col] == unique_vals[0]).astype(int)
            else:
                # if not 0/1, map to binary
                unique_vals = df[col].unique()
                df[col] = (df[col] == unique_vals[0]).astype(int)
        else:
            # for other binary medical variables: >0 = 1, 0 = 0
            df[col] = (df[col > 0]).astype(int)


--- Ensuring 0/1 Encoding Consistency ---
sex: Already encoded as 0/1
fasting_blood_sugar_gt_120_mg_per_dl: Already encoded as 0/1
exercise_induced_angina: Already encoded as 0/1


In [23]:
# verfiy binary encoding
print("\n--- Binary Encoding Verfication ---")
verification_results =  []
for col in binary_cols:
    unique_vals = sorted(df[col].dropna().unique())
    is_binary_01 = set(unique_vals) == {0,1}
    verification_results.append({
        'Column' : col,
        'Unique Values': unique_vals,
        'Is 0/1': is_binary_01,
        'Data Type': df[col].dtype
    })

verification_df = pd.DataFrame(verification_results)
print(verification_df.to_string(index= False))


--- Binary Encoding Verfication ---
                              Column Unique Values  Is 0/1 Data Type
                                 sex        [0, 1]    True     int64
fasting_blood_sugar_gt_120_mg_per_dl        [0, 1]    True     int64
             exercise_induced_angina        [0, 1]    True     int64


In [24]:
# PART 5: FINAL DATASET SUMMARY
print("\n" + "="*60)
print("FINAL ENCODED DATASET SUMMARY")
print("="*60)

# Create final encoded dataset
df_encoded = df.copy()

# Add thal one-hot encoding (recommended approach)
thal_dummies = pd.get_dummies(df_encoded['thal_cleaned'], prefix='thal')
df_encoded = pd.concat([df_encoded, thal_dummies], axis=1)

# Keep ordinal encoding for chest pain
# (already added as 'chest_pain_encoded')

print("\nEncoded Features added to dataset:")
encoded_features = []
for col in df_encoded.columns:
    if col.startswith('thal_') and col != 'thal' and col != 'thal_cleaned':
        encoded_features.append(col)
    elif col == 'chest_pain_encoded':
        encoded_features.append(col)

print(f" -{len(encoded_features)} encoded features:")
for f in encoded_features:
    print(f'{f}')

print(f"\nFinal dataset shape: {df_encoded.shape}")
print(f"Orginial columns: {len(df.columns)}")
print(f"Added encoded Columns: {len(encoded_features)}")


FINAL ENCODED DATASET SUMMARY

Encoded Features added to dataset:
 -4 encoded features:
chest_pain_encoded
thal_Other
thal_normal
thal_reversible_defect

Final dataset shape: (180, 19)
Orginial columns: 16
Added encoded Columns: 4


In [ ]:
import pandas as pd

# Load data
df = pd.read_csv(r"../Data/values.csv")

binary_cols = [
    'sex',
    'fasting_blood_sugar',
    'exercise_induced_angina'
]
binary_cols = [col for col in binary_cols if col in df.columns]

df[binary_cols] = df[binary_cols].apply(pd.to_numeric, errors='coerce')

df[binary_cols] = (df[binary_cols] > 0).astype(int)

print("Binary columns (first 5 rows):")
print(df[binary_cols].head())

print("\nUnique values count:")
print(df[binary_cols].nunique())


Binary columns (first 5 rows):
   sex  exercise_induced_angina
0    1                        0
1    0                        0
2    1                        1
3    1                        0
4    1                        0

Unique values count:
sex                        2
exercise_induced_angina    2
dtype: int64


In [25]:
#Show sample of final encoded data
print("\n--- Sample Encoded Data (First 5 rows)")
sample_cols = ['thal','thal_cleaned'] + [col for col in df_encoded.columns 
    if col.startswith('thal_') and col not in ['thal','thal_cleaned']]
sample_cols += ['chest_pain_type','chest_pain_encoded']
sample_cols += binary_cols

print(df_encoded[sample_cols].head())


--- Sample Encoded Data (First 5 rows)
                thal       thal_cleaned  thal_Other  thal_normal  \
0             normal             normal       False         True   
1             normal             normal       False         True   
2             normal             normal       False         True   
3  reversible_defect  reversible_defect       False        False   
4  reversible_defect  reversible_defect       False        False   

   thal_reversible_defect  chest_pain_type  chest_pain_encoded  sex  \
0                   False                2                   1    1   
1                   False                3                   2    0   
2                   False                4                   3    1   
3                    True                4                   3    1   
4                    True                1                   0    1   

   fasting_blood_sugar_gt_120_mg_per_dl  exercise_induced_angina  
0                                     0                  

In [26]:
# PART 6: ENCODING STRATEGY SUMMARY & RECOMMENDATIONS
print("\n" + "="*60)
print("ENCODING STRATEGY SUMMARY & RECOMMENDATIONS")
print("="*60)

print("""
SUMMARY OF ENCODING DECISIONS:

1. THAL (Blood flow defects):
   - Strategy: One-hot encoding (recommended)
   - Why: No inherent ordinal relationship between defect types
   - Implementation: Created thal_normal, thal_fixed_defect, thal_reversible_defect
   - Alternative: Ordinal encoding available for comparison (thal_ordinal)

2. CHEST PAIN TYPE:
   - Strategy: Ordinal encoding
   - Why: Clear medical severity progression exists
   - Mapping: 0=typical angina → 3=asymptomatic (most severe)
   - Medical justification: Asymptomatic CAD is clinically most concerning

3. BINARY FEATURES:
   - Strategy: Standardized to 0/1 encoding
   - Columns processed: sex, fasting_blood_sugar_gt_120_mg_per_dl, exercise_induced_angina
   - Verification: All confirmed as proper binary (0/1) variables

RECOMMENDATIONS FOR MODELING:
- Use one-hot encoded thal features for models
- Use ordinally encoded chest pain for tree-based models
- Binary features are ready for immediate use
- Consider interaction terms between chest pain and exercise-induced angina

VALIDATION PERFORMED:
- All binary features verified as 0/1
- No rare categories required merging for thal
- Medical justification provided for all encoding decisions
- Both encoding methods implemented for thal as requested
""")

# %%
# Save the encoded dataset
output_path = "../Data/values_encoded.csv"
df_encoded.to_csv(output_path, index=False)
print(f"\nEncoded dataset saved to: {output_path}")


ENCODING STRATEGY SUMMARY & RECOMMENDATIONS

SUMMARY OF ENCODING DECISIONS:

1. THAL (Blood flow defects):
   - Strategy: One-hot encoding (recommended)
   - Why: No inherent ordinal relationship between defect types
   - Implementation: Created thal_normal, thal_fixed_defect, thal_reversible_defect
   - Alternative: Ordinal encoding available for comparison (thal_ordinal)

2. CHEST PAIN TYPE:
   - Strategy: Ordinal encoding
   - Why: Clear medical severity progression exists
   - Mapping: 0=typical angina → 3=asymptomatic (most severe)
   - Medical justification: Asymptomatic CAD is clinically most concerning

3. BINARY FEATURES:
   - Strategy: Standardized to 0/1 encoding
   - Columns processed: sex, fasting_blood_sugar_gt_120_mg_per_dl, exercise_induced_angina
   - Verification: All confirmed as proper binary (0/1) variables

RECOMMENDATIONS FOR MODELING:
- Use one-hot encoded thal features for models
- Use ordinally encoded chest pain for tree-based models
- Binary features are re